# Train XGBoost Model

### Purpose

**This notebook develops and evaluates an XGBoost model on concatenated RNA and clinical features as a nonlinear benchmark against the logistic regression baselines.**

### Objectives
- Train an XGBoost classifier on combined RNA and clinical features.
- Select hyperparameters via cross-validation on the training set only.
- Evaluate discrimination on validation and test splits.
- Compute SHAP feature importances to identify key predictors.
- Prototype modeling and evaluation steps before implementation in the pipeline.

### Workflow

1. Load and inspect assembled datasets
   - Load preprocessed feature matrices and outcome labels produced by the data assembly step.
   - Load `X_concat.parquet` (pre-assembled concatenated clinical + RNA features) for each split.
   - Confirm expected shapes and verify sample alignment between features and outcome labels.

**Subsequent exploratory steps use the training and validation data only.**

2. Tune hyperparameters via cross-validation
   - Define a hyperparameter search grid (e.g., `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`, `n_estimators`).
   - Select best parameters via stratified k-fold CV on the training set only.

3. Train final XGBoost model
   - Fit XGBoost on the full training set using selected hyperparameters.
   - Use early stopping on a held-out eval set from within training to determine `n_estimators`.

4. Generate validation predictions
   - Produce predicted probabilities for the validation set.
   - Evaluate discrimination metrics.

5. Evaluate model performance
   - Compute ROC-AUC and average precision (AP) as primary discrimination metrics.
   - Generate calibration curve as diagnostic check.
   - Compare against logistic regression baselines.

6. Evaluate risk-tier separation
   - Bin patients into three tiers: top 20% (high-risk), middle 60%, bottom 20% (low-risk).
   - Examine outcome rates across tiers.

7. SHAP feature importance
   - Compute SHAP values on the validation set.
   - Inspect top features by mean absolute SHAP value.
   - Confirm biologically plausible drivers.

**Now evaluate on test data.**

8. Evaluate final model on the test set
   - Generate test predictions using the trained model.
   - Compute final evaluation metrics.

9. Validate modeling outputs
   - Confirm prediction counts match split sizes.
   - Verify sample alignment across datasets.
   - Verify no data leakage between splits.

10. Test train XGBoost module

## 1. Load and inspect assembled datasets
   - Load preprocessed feature matrices and outcome labels produced by the data assembly step.
   - Load `X_concat.parquet` (pre-assembled concatenated clinical + RNA features) for each split.
   - Confirm expected shapes and verify sample alignment between features and outcome labels.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

assembled_dir = Path("../data/processed/assembled")

# Load concatenated feature matrices and targets for each split
X_train_df = pd.read_parquet(assembled_dir / "train/X_concat.parquet")
X_val_df   = pd.read_parquet(assembled_dir / "val/X_concat.parquet")
X_test_df  = pd.read_parquet(assembled_dir / "test/X_concat.parquet")

y_train = pd.read_parquet(assembled_dir / "train/y.parquet")["y"]
y_val   = pd.read_parquet(assembled_dir / "val/y.parquet")["y"]
y_test  = pd.read_parquet(assembled_dir / "test/y.parquet")["y"]

# Verify alignment between features and targets
for split, X, y in [("train", X_train_df, y_train), ("val", X_val_df, y_val), ("test", X_test_df, y_test)]:
    assert X.index.equals(y.index), f"{split}: index mismatch between X and y"
    print(f"{split}: n={len(y)}, n_events={y.sum()}, features={X.shape[1]}")

# XGBoost does not allow [ ] < in feature names — sanitize before fitting
X_train_df.columns = X_train_df.columns.str.replace(r"[\[\]<]", "_", regex=True)
X_val_df.columns   = X_val_df.columns.str.replace(r"[\[\]<]", "_", regex=True)
X_test_df.columns  = X_test_df.columns.str.replace(r"[\[\]<]", "_", regex=True)

display(X_train_df.head(), X_train_df.tail())


train: n=203, n_events=66, features=25529
val: n=43, n_events=14, features=25529
test: n=44, n_events=14, features=25529


,year_of_birth.demographic,year_of_diagnosis.diagnoses,age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,days_to_collection.samples,initial_weight.samples,oct_embedded.samples,ajcc_pathologic_m.diagnoses_M0,ajcc_pathologic_m.diagnoses_M1,ajcc_pathologic_m.diagnoses_MX,ajcc_pathologic_m.diagnoses_cM0 (i+),...,ENSG00000288573.1,ENSG00000288585.1,ENSG00000288586.1,ENSG00000288596.2,ENSG00000288600.1,ENSG00000288605.1,ENSG00000288612.1,ENSG00000288658.1,ENSG00000288670.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-AR-A1AN-01A,1960.0,2006.0,46.504110,1463.0,70.0,1,1.0,0.0,0.0,0.0,...,-1.353641,-0.352161,0.247873,-1.847424,-0.651507,-0.606533,-0.440411,0.176202,0.426529,1.225862
TCGA-B6-A2IU-01A,1936.0,1998.0,62.753425,4665.0,140.0,1,1.0,0.0,0.0,0.0,...,0.012429,1.103978,1.517371,0.609031,0.524253,-0.474512,-0.202529,-0.660575,0.394027,-0.961233
TCGA-C8-A12Q-01A,1932.0,2010.0,78.720548,57.0,810.0,1,1.0,0.0,0.0,0.0,...,-1.509343,-0.020435,-0.527875,-0.585608,0.204504,-0.378965,2.682691,-0.551112,0.489490,-1.008443
TCGA-BH-A0E9-01B,1953.0,2006.0,53.191781,1357.0,480.0,1,1.0,0.0,0.0,0.0,...,0.585732,2.531337,-0.092806,0.688908,-0.607627,-0.034685,1.212378,-0.406525,0.965263,0.145407
TCGA-A2-A0ST-01A,1941.0,2003.0,62.800000,2597.0,210.0,1,1.0,0.0,0.0,0.0,...,0.178092,-0.914768,-1.051409,-0.765611,2.093778,-0.025683,-0.928029,0.542307,-1.556974,-0.272568


,year_of_birth.demographic,year_of_diagnosis.diagnoses,age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,days_to_collection.samples,initial_weight.samples,oct_embedded.samples,ajcc_pathologic_m.diagnoses_M0,ajcc_pathologic_m.diagnoses_M1,ajcc_pathologic_m.diagnoses_MX,ajcc_pathologic_m.diagnoses_cM0 (i+),...,ENSG00000288573.1,ENSG00000288585.1,ENSG00000288586.1,ENSG00000288596.2,ENSG00000288600.1,ENSG00000288605.1,ENSG00000288612.1,ENSG00000288658.1,ENSG00000288670.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-GM-A2DM-01A,1947.0,2004.0,57.972603,2464.0,120.0,0,1.0,0.0,0.0,0.0,...,0.265048,-0.884606,0.141415,-0.480243,-0.579166,-0.528830,-0.286348,-0.517218,0.045952,0.367215
TCGA-AR-A2LE-01A,1932.0,2001.0,69.860274,3755.0,320.0,1,1.0,0.0,0.0,0.0,...,0.236000,-0.231015,0.883146,-0.100171,-0.352117,-0.606533,-0.897766,3.383043,0.066095,-0.709205
TCGA-E2-A1LK-01A,1924.0,2008.0,84.265753,1051.0,70.0,0,1.0,0.0,0.0,0.0,...,0.110951,-0.607541,0.610262,1.347791,-0.521076,1.184644,1.959450,1.703494,0.349865,-0.400755
TCGA-E9-A1RB-01A,1971.0,2011.0,40.424658,39.0,490.0,1,1.0,0.0,0.0,0.0,...,-1.007468,-0.571601,-1.173723,0.221939,-0.343172,-0.560550,-0.506072,-0.084724,0.648266,-0.360840
TCGA-B6-A0I6-01A,1941.0,1990.0,49.353425,7362.0,200.0,1,1.0,0.0,0.0,0.0,...,0.366692,-0.945813,2.813252,0.383612,-0.208193,0.491286,-1.132417,1.301828,-0.865487,-1.733192


## 2. Tune hyperparameters via cross-validation
   - Define a hyperparameter search grid (e.g., `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`, `n_estimators`).
   - Select best parameters via stratified k-fold CV on the training set only.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier

param_grid = {
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.05, 0.1, 0.3],  # low: clinical features are a tiny fraction of 25k+ columns
    "min_child_weight": [1, 3, 5],
}

xgb = XGBClassifier(
    n_estimators=100,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=1,
)

# Search over param_grid using stratified 5-fold CV on training data only
search = RandomizedSearchCV(
    xgb,
    param_grid,
    n_iter=30,
    scoring="roc_auc",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train_df, y_train)

best_params = search.best_params_
print(f"Best CV AUC: {search.best_score_:.3f}")
print(f"Best params: {best_params}")


Fitting 5 folds for each of 30 candidates, totalling 150 fits



3. Train final XGBoost model
   - Fit XGBoost on the full training set using selected hyperparameters.
   - Use early stopping on a held-out eval set from within training to determine `n_estimators`.


## 4. Generate validation predictions
   - Produce predicted probabilities for the validation set.
   - Evaluate discrimination metrics.

## 5. Evaluate model performance
   - Compute ROC-AUC and average precision (AP) as primary discrimination metrics.
   - Generate calibration curve as diagnostic check.
   - Compare against logistic regression baselines.


6. Evaluate risk-tier separation
   - Bin patients into three tiers: top 20% (high-risk), middle 60%, bottom 20% (low-risk).
   - Examine outcome rates across tiers.

7. SHAP feature importance
   - Compute SHAP values on the validation set.
   - Inspect top features by mean absolute SHAP value.
   - Confirm biologically plausible drivers.

**Now evaluate on test data.**

8. Evaluate final model on the test set
   - Generate test predictions using the trained model.
   - Compute final evaluation metrics.

9. Validate modeling outputs
   - Confirm prediction counts match split sizes.
   - Verify sample alignment across datasets.
   - Verify no data leakage between splits.

10. Test train XGBoost module